# Real-time / near-real-time 2-photon GCaMP analysis pipeline

This notebook watches a microscope output directory for newly completed recording folders. For each new folder, it:

1. Loads an image series of individual TIFF frames.
2. Builds an average intensity projection.
3. Detects layer 2/3 pyramidal neuron soma from the average projection using ML segmentation.
4. Extracts fluorescence traces from every detected soma mask.
5. Converts raw fluorescence to ΔF/F.
6. Screens cells as responders or non-responders using stimulus timing windows.
7. Writes CSV/XLSX outputs and saves an annotated average projection with numbered, color-coded cells.

**Assumptions:** frames are 256 × 256 single-channel TIFFs, acquired at 30 Hz, one folder per recording. If your microscope writes multi-page TIFFs or OME-TIFFs, adjust `list_tiff_files()` / `read_frame()` accordingly.

## Library roles

- `pathlib`, `time`, `json`, `logging`: file paths, folder monitoring, metadata, and progress messages.
- `numpy`, `pandas`: numerical arrays, traces, ΔF/F, and tables.
- `tifffile`: fast TIFF reading/writing.
- `scikit-image`: image normalization, morphology cleanup, mask measurements, and annotation support.
- `matplotlib`: saving annotated projections and QC plots.
- `watchdog`: event-based folder watching so new microscope folders trigger analysis.
- `cellpose`: pretrained ML instance segmentation for soma detection.
- `openpyxl`: Excel export through `pandas.to_excel()`.

Install once in your environment if needed:

```bash
pip install numpy pandas tifffile scikit-image matplotlib watchdog cellpose openpyxl
```

## Why this ML model?

The default model here is **Cellpose / Cellpose-SAM-style instance segmentation** applied to the average GCaMP projection. This is a good first choice because it is a generalist pretrained cell/object segmentation model, returns separate masks for individual cells, can be run directly from Python, and can be fine-tuned later if your GCaMP soma images differ from the pretrained distribution.

For a production calcium-imaging pipeline, **Suite2p** and **CaImAn/OnACID** are also strong options. Suite2p is purpose-built for two-photon calcium imaging and includes registration, ROI detection, signal extraction, ROI classification, and spike detection. CaImAn provides scalable calcium-imaging processing and has online analysis support through OnACID. In this notebook, I use Cellpose because your requested workflow starts from an average projection and asks for direct soma masks from that image; Cellpose is simple and modular for that step. If motion correction and neuropil correction become central, Suite2p or CaImAn would be better production backbones.

In [ ]:
# Core imports
from pathlib import Path
import time
import json
import logging
from datetime import datetime

import numpy as np
import pandas as pd
import tifffile as tiff

from skimage import exposure, measure, morphology, segmentation, color
from skimage.util import img_as_float32

import matplotlib.pyplot as plt

# Folder watching
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

# Optional ML segmentation backend
try:
    from cellpose import models
    CELLPOSE_AVAILABLE = True
except Exception as e:
    CELLPOSE_AVAILABLE = False
    print("Cellpose import failed. Segmentation will use fallback thresholding. Error:", e)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

In [ ]:
# -------------------------
# User configuration
# -------------------------

ROOT_WATCH_DIR = Path("/path/to/microscope/output")  # folder where microscope creates new recording folders
OUTPUT_ROOT = Path("/path/to/analysis/output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

FRAME_RATE_HZ = 30.0
IMAGE_SHAPE = (256, 256)
TIFF_GLOB = "*.tif*"

# Folder completion logic: analyze only after file count has stopped changing.
STABLE_SECONDS = 15
POLL_SECONDS = 2

# Cellpose settings. Adjust DIAMETER_PX based on expected soma diameter in pixels.
USE_CELLPOSE = True
CELLPOSE_MODEL_TYPE = "cyto3"  # common choices: "cyto3", "cyto", "nuclei" depending on installed version
DIAMETER_PX = 12             # L2/3 soma example; adjust after QC
CELLPOSE_FLOW_THRESHOLD = 0.4
CELLPOSE_CELLPROB_THRESHOLD = 0.0

# Mask cleanup
MIN_SOMA_AREA_PX = 30
MAX_SOMA_AREA_PX = 600

# ΔF/F settings
F0_PERCENTILE = 20           # robust baseline if no explicit baseline epochs are supplied
SMOOTH_WINDOW_FRAMES = 3     # light smoothing; set 1 for none

# Responder detection
RESPONSE_THRESHOLD_DFF = 0.20      # absolute ΔF/F threshold
RESPONSE_Z_THRESHOLD = 3.0         # response must exceed baseline by this many SDs
MIN_RESPONSE_FRAMES = 3            # require at least this many frames above threshold in a stimulus window

# Optional stimulus timing file inside each recording folder.
# Expected columns: protocol, onset_s, offset_s. If missing, responder screening uses whole-trace heuristic.
STIMULUS_CSV_NAME = "stimulus_times.csv

## Helper functions

These functions keep the workflow modular: file discovery, folder-completion checks, projection generation, ML segmentation, trace extraction, ΔF/F conversion, responder classification, and saving outputs.

In [ ]:
def list_tiff_files(folder: Path) -> list[Path]:
    """Return sorted TIFF frames in a recording folder."""
    return sorted(folder.glob(TIFF_GLOB))


def read_frame(path: Path) -> np.ndarray:
    """Read one TIFF frame as float32."""
    arr = tiff.imread(str(path))
    if arr.ndim > 2:
        # If an unexpected channel/page dimension exists, take the first plane.
        arr = arr[0]
    return arr.astype(np.float32)


def wait_until_folder_stable(folder: Path, stable_seconds: int = STABLE_SECONDS) -> bool:
    """Wait until no new TIFFs appear for stable_seconds."""
    last_count = -1
    last_change = time.time()
    while True:
        n = len(list_tiff_files(folder))
        if n != last_count:
            last_count = n
            last_change = time.time()
        if n > 0 and (time.time() - last_change) >= stable_seconds:
            return True
        time.sleep(POLL_SECONDS)


def make_average_projection(tiff_files: list[Path]) -> np.ndarray:
    """Compute average intensity projection without loading the whole movie into RAM."""
    running_sum = None
    for i, f in enumerate(tiff_files):
        frame = read_frame(f)
        if running_sum is None:
            running_sum = np.zeros_like(frame, dtype=np.float64)
        running_sum += frame
    return (running_sum / len(tiff_files)).astype(np.float32)


def normalize_for_segmentation(avg_img: np.ndarray) -> np.ndarray:
    """Normalize contrast for segmentation."""
    img = img_as_float32(avg_img)
    p1, p99 = np.percentile(img, [1, 99.7])
    img = exposure.rescale_intensity(img, in_range=(p1, p99), out_range=(0, 1))
    return img.astype(np.float32)

In [ ]:
def segment_soma_cellpose(avg_img: np.ndarray) -> np.ndarray:
    """Segment soma instances using Cellpose. Returns label image: 0=background, 1..N=cells."""
    img = normalize_for_segmentation(avg_img)
    model = models.CellposeModel(model_type=CELLPOSE_MODEL_TYPE)
    masks, flows, styles = model.eval(
        img,
        channels=[0, 0],
        diameter=DIAMETER_PX,
        flow_threshold=CELLPOSE_FLOW_THRESHOLD,
        cellprob_threshold=CELLPOSE_CELLPROB_THRESHOLD,
    )
    return masks.astype(np.int32)


def segment_soma_fallback(avg_img: np.ndarray) -> np.ndarray:
    """Fallback segmentation if Cellpose is unavailable. This is not ML; use only for testing."""
    img = normalize_for_segmentation(avg_img)
    thresh = np.percentile(img, 92)
    bw = img > thresh
    bw = morphology.remove_small_objects(bw, MIN_SOMA_AREA_PX)
    bw = morphology.binary_opening(bw, morphology.disk(1))
    labels = measure.label(bw)
    return labels.astype(np.int32)


def clean_label_masks(labels: np.ndarray) -> np.ndarray:
    """Remove objects that are too small/large and relabel consecutively."""
    cleaned = np.zeros_like(labels, dtype=np.int32)
    next_id = 1
    for region in measure.regionprops(labels):
        if MIN_SOMA_AREA_PX <= region.area <= MAX_SOMA_AREA_PX:
            cleaned[labels == region.label] = next_id
            next_id += 1
    return cleaned


def segment_soma(avg_img: np.ndarray) -> np.ndarray:
    """Main segmentation wrapper."""
    if USE_CELLPOSE and CELLPOSE_AVAILABLE:
        labels = segment_soma_cellpose(avg_img)
    else:
        labels = segment_soma_fallback(avg_img)
    return clean_label_masks(labels)

In [ ]:
def extract_traces_from_masks(tiff_files: list[Path], labels: np.ndarray) -> pd.DataFrame:
    """Extract mean fluorescence per cell mask for every frame."""
    cell_ids = np.array(sorted([x for x in np.unique(labels) if x != 0]), dtype=int)
    n_cells = len(cell_ids)
    n_frames = len(tiff_files)
    traces = np.zeros((n_frames, n_cells), dtype=np.float32)

    mask_pixels = [(labels == cid) for cid in cell_ids]

    for frame_idx, f in enumerate(tiff_files):
        frame = read_frame(f)
        for j, mask in enumerate(mask_pixels):
            traces[frame_idx, j] = frame[mask].mean()

    time_s = np.arange(n_frames) / FRAME_RATE_HZ
    df = pd.DataFrame(traces, columns=[f"cell_{cid:03d}" for cid in cell_ids])
    df.insert(0, "time_s", time_s)
    return df


def smooth_traces(df: pd.DataFrame, window: int = SMOOTH_WINDOW_FRAMES) -> pd.DataFrame:
    """Light rolling smoothing for fluorescence traces."""
    if window <= 1:
        return df.copy()
    out = df.copy()
    cols = [c for c in out.columns if c.startswith("cell_")]
    out[cols] = out[cols].rolling(window=window, center=True, min_periods=1).mean()
    return out


def compute_dff(trace_df: pd.DataFrame, f0_percentile: float = F0_PERCENTILE) -> pd.DataFrame:
    """Compute ΔF/F for each cell using a robust percentile baseline."""
    out = trace_df.copy()
    cols = [c for c in out.columns if c.startswith("cell_")]
    F = out[cols].to_numpy(dtype=np.float32)
    F0 = np.percentile(F, f0_percentile, axis=0)
    F0 = np.maximum(F0, 1e-6)
    dff = (F - F0) / F0
    out[cols] = dff
    return out

In [ ]:
def load_stimulus_table(folder: Path) -> pd.DataFrame | None:
    """Load optional stimulus table with protocol, onset_s, offset_s."""
    stim_path = folder / STIMULUS_CSV_NAME
    if not stim_path.exists():
        return None
    stim = pd.read_csv(stim_path)
    required = {"protocol", "onset_s", "offset_s"}
    missing = required - set(stim.columns)
    if missing:
        raise ValueError(f"Stimulus table is missing columns: {missing}")
    return stim


def classify_responders(dff_df: pd.DataFrame, stim: pd.DataFrame | None) -> pd.DataFrame:
    """Classify each cell as responder/non-responder. Uses stimulus windows if available."""
    time_s = dff_df["time_s"].to_numpy()
    cell_cols = [c for c in dff_df.columns if c.startswith("cell_")]
    rows = []

    for col in cell_cols:
        y = dff_df[col].to_numpy(dtype=np.float32)

        if stim is not None and len(stim) > 0:
            # Baseline = all frames outside stimulus windows.
            stim_mask = np.zeros_like(time_s, dtype=bool)
            for _, r in stim.iterrows():
                stim_mask |= (time_s >= r["onset_s"]) & (time_s <= r["offset_s"])
            baseline = y[~stim_mask] if np.any(~stim_mask) else y
            base_mean = float(np.mean(baseline))
            base_sd = float(np.std(baseline) + 1e-6)

            best_protocol = None
            best_peak = -np.inf
            best_z = -np.inf
            responder = False

            for _, r in stim.iterrows():
                win = (time_s >= r["onset_s"]) & (time_s <= r["offset_s"])
                yw = y[win]
                if len(yw) == 0:
                    continue
                peak = float(np.max(yw))
                z = float((peak - base_mean) / base_sd)
                above = (yw >= RESPONSE_THRESHOLD_DFF) & (yw >= base_mean + RESPONSE_Z_THRESHOLD * base_sd)
                is_resp = bool(np.sum(above) >= MIN_RESPONSE_FRAMES)
                if peak > best_peak:
                    best_peak = peak
                    best_z = z
                    best_protocol = r["protocol"]
                responder = responder or is_resp

        else:
            # Fallback when no stimulus file exists: whole-trace thresholding.
            base_mean = float(np.percentile(y, 20))
            base_sd = float(np.std(y[y <= np.percentile(y, 50)]) + 1e-6)
            best_peak = float(np.max(y))
            best_z = float((best_peak - base_mean) / base_sd)
            best_protocol = "unknown_no_stimulus_file"
            responder = bool((best_peak >= RESPONSE_THRESHOLD_DFF) and (best_z >= RESPONSE_Z_THRESHOLD))

        rows.append({
            "cell": col,
            "responder": responder,
            "best_protocol": best_protocol,
            "peak_dff": best_peak,
            "peak_z": best_z,
            "baseline_mean_dff": base_mean,
            "baseline_sd_dff": base_sd,
        })

    return pd.DataFrame(rows)

In [ ]:
def build_cell_metadata(labels: np.ndarray, responder_df: pd.DataFrame) -> pd.DataFrame:
    """Create per-cell mask metadata: centroid, area, and responder label."""
    rows = []
    resp_map = dict(zip(responder_df["cell"], responder_df["responder"]))
    for region in measure.regionprops(labels):
        cid = int(region.label)
        cname = f"cell_{cid:03d}"
        y, x = region.centroid
        rows.append({
            "cell": cname,
            "cell_id": cid,
            "centroid_x_px": x,
            "centroid_y_px": y,
            "area_px": int(region.area),
            "responder": bool(resp_map.get(cname, False)),
        })
    return pd.DataFrame(rows).merge(responder_df, on=["cell", "responder"], how="left")


def save_annotated_projection(avg_img: np.ndarray, labels: np.ndarray, metadata: pd.DataFrame, out_path: Path):
    """Save average projection with cell numbers and responder color coding."""
    img = normalize_for_segmentation(avg_img)
    boundaries = segmentation.find_boundaries(labels, mode="outer")

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(img, cmap="gray")

    # Responders = green text/outline; non-responders = red text/outline.
    for _, row in metadata.iterrows():
        cid = int(row["cell_id"])
        mask_boundary = boundaries & (labels == cid)
        ys, xs = np.where(mask_boundary)
        color_name = "lime" if row["responder"] else "red"
        if len(xs) > 0:
            ax.scatter(xs, ys, s=0.2, c=color_name)
        ax.text(row["centroid_x_px"], row["centroid_y_px"], str(cid),
                color=color_name, fontsize=8, ha="center", va="center")

    ax.set_title("Detected soma: green=responder, red=non-responder")
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(out_path, dpi=300)
    plt.close(fig)


def save_trace_qc(dff_df: pd.DataFrame, responder_df: pd.DataFrame, out_path: Path, max_cells: int = 30):
    """Save a compact QC plot of ΔF/F traces."""
    cols = [c for c in dff_df.columns if c.startswith("cell_")][:max_cells]
    fig, ax = plt.subplots(figsize=(10, 5))
    for col in cols:
        ax.plot(dff_df["time_s"], dff_df[col], linewidth=0.8, alpha=0.7)
    ax.axhline(RESPONSE_THRESHOLD_DFF, linestyle="--", linewidth=1)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("ΔF/F")
    ax.set_title(f"ΔF/F QC traces, first {len(cols)} cells")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

In [ ]:
def analyze_recording_folder(folder: Path) -> dict:
    """Run full pipeline on one completed microscope recording folder."""
    folder = Path(folder)
    logging.info(f"Analyzing recording folder: {folder}")

    tiff_files = list_tiff_files(folder)
    if len(tiff_files) == 0:
        raise FileNotFoundError(f"No TIFF files found in {folder}")

    recording_id = folder.name
    out_dir = OUTPUT_ROOT / recording_id
    out_dir.mkdir(parents=True, exist_ok=True)

    avg_img = make_average_projection(tiff_files)
    tiff.imwrite(out_dir / "average_projection.tif", avg_img.astype(np.float32))

    labels = segment_soma(avg_img)
    tiff.imwrite(out_dir / "soma_labels.tif", labels.astype(np.uint16))

    raw_traces = extract_traces_from_masks(tiff_files, labels)
    raw_traces.to_csv(out_dir / "raw_fluorescence_traces.csv", index=False)

    smooth_raw = smooth_traces(raw_traces)
    dff = compute_dff(smooth_raw)
    dff.to_csv(out_dir / "dff_traces.csv", index=False)

    stim = load_stimulus_table(folder)
    if stim is not None:
        stim.to_csv(out_dir / "stimulus_times_used.csv", index=False)

    responder_df = classify_responders(dff, stim)
    metadata = build_cell_metadata(labels, responder_df)

    metadata.to_csv(out_dir / "cell_summary.csv", index=False)
    with pd.ExcelWriter(out_dir / "analysis_summary.xlsx", engine="openpyxl") as writer:
        metadata.to_excel(writer, sheet_name="cell_summary", index=False)
        responder_df.to_excel(writer, sheet_name="responders", index=False)
        dff.to_excel(writer, sheet_name="dff_traces", index=False)
        raw_traces.to_excel(writer, sheet_name="raw_traces", index=False)
        if stim is not None:
            stim.to_excel(writer, sheet_name="stimulus_times", index=False)

    save_annotated_projection(avg_img, labels, metadata, out_dir / "annotated_average_projection.png")
    save_trace_qc(dff, responder_df, out_dir / "dff_trace_qc.png")

    summary = {
        "recording_id": recording_id,
        "input_folder": str(folder),
        "output_folder": str(out_dir),
        "n_frames": len(tiff_files),
        "frame_rate_hz": FRAME_RATE_HZ,
        "n_detected_cells": int(len(metadata)),
        "n_responders": int(metadata["responder"].sum()) if len(metadata) else 0,
        "completed_at": datetime.now().isoformat(),
    }
    with open(out_dir / "run_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    logging.info(f"Done: {summary}")
    return summary

## Run once on an existing folder

Use this first to validate segmentation diameter, thresholds, and output formatting before enabling automatic folder watching.

In [ ]:
# Example manual run
# test_folder = Path("/path/to/microscope/output/recording_001")
# summary = analyze_recording_folder(test_folder)
# summary

## Automatic folder watching

This section watches `ROOT_WATCH_DIR`. When the microscope creates a new subfolder, the notebook waits until the TIFF count stops changing, then runs the full analysis.

In [ ]:
class NewRecordingHandler(FileSystemEventHandler):
    def __init__(self):
        super().__init__()
        self.seen = set()

    def on_created(self, event):
        if event.is_directory:
            folder = Path(event.src_path)
            if folder in self.seen:
                return
            self.seen.add(folder)
            logging.info(f"New folder detected: {folder}")
            try:
                wait_until_folder_stable(folder)
                analyze_recording_folder(folder)
            except Exception as e:
                logging.exception(f"Analysis failed for {folder}: {e}")


def start_folder_watcher(root_dir: Path = ROOT_WATCH_DIR):
    """Start folder watcher. Stop with observer.stop(); observer.join()."""
    root_dir = Path(root_dir)
    root_dir.mkdir(parents=True, exist_ok=True)
    handler = NewRecordingHandler()
    observer = Observer()
    observer.schedule(handler, str(root_dir), recursive=False)
    observer.start()
    logging.info(f"Watching for new recording folders in: {root_dir}")
    return observer

# Start watching. Keep this cell running in an acquisition session.
# observer = start_folder_watcher(ROOT_WATCH_DIR)

# To stop later:
# observer.stop()
# observer.join()

## Expected input/output structure

Example input folder:

```text
recording_001/
  frame_000001.tif
  frame_000002.tif
  ...
  stimulus_times.csv      # optional but recommended
```

Expected `stimulus_times.csv` format:

```text
protocol,onset_s,offset_s
flash_1,10.0,12.0
grating_0deg,20.0,25.0
grating_90deg,35.0,40.0
```

Output folder:

```text
analysis/output/recording_001/
  average_projection.tif
  soma_labels.tif
  annotated_average_projection.png
  raw_fluorescence_traces.csv
  dff_traces.csv
  cell_summary.csv
  analysis_summary.xlsx
  dff_trace_qc.png
  run_summary.json
```

## Notes for improving the pipeline

1. **Motion correction:** this notebook assumes the field of view is stable enough that an average projection is meaningful. If motion is present, add Suite2p or CaImAn motion correction before projection and trace extraction.
2. **Neuropil correction:** for publication-level calcium analysis, subtract surrounding neuropil fluorescence before ΔF/F. This notebook extracts soma-only mean fluorescence as the first-pass workflow.
3. **Online vs near-real-time:** this notebook analyzes after a recording folder is complete. True frame-by-frame online prediction is possible, but usually requires a streaming interface from the microscope software instead of waiting for TIFF files to finish writing.
4. **Model fine-tuning:** after manually validating/correcting masks from several recordings, fine-tune Cellpose or train a U-Net/Mask R-CNN model on your own L2/3 GCaMP soma images.
5. **Responder rules:** the default classifier is threshold-based on ΔF/F and z-score. Once you have labeled examples, replace `classify_responders()` with a supervised model such as XGBoost, random forest, or a small temporal CNN using features from stimulus-locked traces.